In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from tqdm import tqdm

In [7]:
# --- Hyperparameters ---
NUM_INPUT = 784
NUM_OUTPUT = 50
DT = 0.005         # 5 ms time step
T_MAX = 0.350      # 350 ms per image
EPOCHS = 1

In [3]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')

In [13]:
images = mnist.data[:10000] / 255.0
    
# Initialize weights U(0, 1)
weights = np.random.uniform(0, 1, (NUM_INPUT, NUM_OUTPUT))

# Pre-allocate neuron states
v_pre = np.zeros(NUM_INPUT)
v_post = np.zeros(NUM_OUTPUT)
n_adapt = np.zeros(NUM_OUTPUT)

refractory_pre = np.zeros(NUM_INPUT)
refractory_post = np.zeros(NUM_OUTPUT)
wta_clamp = np.zeros(NUM_OUTPUT)

In [14]:
learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]

In [15]:
for LR in learning_rates:
    print(f'leaning rate = {LR}')
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}:")
        for img_idx, img in tqdm(enumerate(images), total=len(images), desc="training: "):
            # Reset states for new image
            v_pre.fill(0.0)
            v_post.fill(0.0)
            
            # Input current proportional to pixel intensity
            I_in = img 
            
            for t in np.arange(0, T_MAX, DT):
                # --- Update Input Neurons (LIF) ---
                active_pre = refractory_pre <= 0
                v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + I_in[active_pre] + 0.5)
                
                spikes_pre = v_pre >= 1.0
                v_pre[spikes_pre] = -1.0
                refractory_pre[spikes_pre] = 0.005
                refractory_pre -= DT
                
                # --- Update Output Neurons (ALIF) ---
                active_post = (refractory_post <= 0) & (wta_clamp <= 0)
                I_post = spikes_pre.astype(float) @ weights
                
                v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
                n_adapt -= (DT / 1.0) * n_adapt
                
                spikes_post = v_post >= 1.0
                
                # --- Apply WTA and VDSP ---
                if spikes_post.any():
                    winner_idx = np.argmax(v_post) # Winner takes all
                    v_post[winner_idx] = 0.0
                    refractory_post[winner_idx] = 0.005
                    n_adapt[winner_idx] += 0.01
                    
                    # Clamp others
                    wta_clamp[:] = 0.010
                    wta_clamp[winner_idx] = 0.0
                    v_post[wta_clamp > 0] = 0.0
                    
                    # VDSP Weight Update for winner
                    v_pre_winner = v_pre
                    
                    # Potentiation (V_pre < 0)
                    pot_mask = v_pre_winner < 0
                    weights[pot_mask, winner_idx] += LR * (1.0 - weights[pot_mask, winner_idx]) * np.abs(v_pre_winner[pot_mask] + 1)
                    
                    # Depression (V_pre > 0)
                    dep_mask = v_pre_winner > 0
                    weights[dep_mask, winner_idx] -= LR * weights[dep_mask, winner_idx] * np.abs(v_pre_winner[dep_mask])
                    
                    # Clip weights (safety bound)
                    weights[:, winner_idx] = np.clip(weights[:, winner_idx], 0, 1)

                refractory_post -= DT
                wta_clamp -= DT
                                
        np.save(f'vdsp_weights_{LR}_{epoch}.npy', weights)
        print(f"Weights saved: vdsp_weights_{LR}_{epoch}.npy")

leaning rate = 0.0001
Epoch 1:


training: 100%|█████████████████████| 10000/10000 [01:24<00:00, 118.97it/s]


Weights saved: vdsp_weights_0.0001_0.npy
leaning rate = 0.0005
Epoch 1:


training: 100%|█████████████████████| 10000/10000 [01:21<00:00, 122.51it/s]


Weights saved: vdsp_weights_0.0005_0.npy
leaning rate = 0.001
Epoch 1:


...

Weights saved: vdsp_weights_0.01_0.npy
leaning rate = 0.05
Epoch 1:


training: 100%|█████████████████████| 10000/10000 [01:03<00:00, 156.63it/s]


Weights saved: vdsp_weights_0.05_0.npy
leaning rate = 0.1
Epoch 1:


training: 100%|█████████████████████| 10000/10000 [01:04<00:00, 154.13it/s]

Weights saved: vdsp_weights_0.1_0.npy


In [16]:
images = mnist.data[:5000] / 255.0
labels = mnist.target[:5000].astype(int)
epoch = 0
for LR in learning_rates:
    weights = np.load(f'vdsp_weights_{LR}_{epoch}.npy')
    spike_counts = np.zeros((NUM_OUTPUT, 10))
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    
    for img_idx, img in tqdm(enumerate(images), total=len(images), desc="images"):
        v_pre.fill(0.0)
        v_post.fill(0.0)
        img_label = labels[img_idx]
        
        for t in np.arange(0, T_MAX, DT):
            v_pre += (DT / 0.03) * (-v_pre + img + 0.5)
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            
            v_post += (DT / 0.03) * (-v_post + (spikes_pre.astype(float) @ weights))
            spikes_post = v_post >= 1.0
            
            if spikes_post.any():
                fired_neurons = np.where(spikes_post)[0]
                for winner in fired_neurons:
                    spike_counts[winner, img_label] += 1
                v_post[spikes_post] = 0.0
                
    assigned_labels = np.argmax(spike_counts, axis=1)
    np.save(f'assigned_labels_{LR}_{epoch}.npy', assigned_labels)
    print(f"labels saved: assigned_labels_{LR}_{epoch}.npy")

images: 100%|█████████████████████████| 5000/5000 [00:20<00:00, 240.80it/s]


labels saved: assigned_labels_0.0001_0.npy


images: 100%|█████████████████████████| 5000/5000 [00:18<00:00, 264.46it/s]


labels saved: assigned_labels_0.0005_0.npy


images: 100%|█████████████████████████| 5000/5000 [00:18<00:00, 273.56it/s]


...

labels saved: assigned_labels_0.01_0.npy


images: 100%|█████████████████████████| 5000/5000 [00:16<00:00, 302.77it/s]


labels saved: assigned_labels_0.05_0.npy


images: 100%|█████████████████████████| 5000/5000 [00:15<00:00, 312.69it/s]

labels saved: assigned_labels_0.1_0.npy


In [18]:
images = mnist.data[60000:] / 255.0
true_labels = mnist.target[60000:].astype(int)
epoch = 0

for LR in learning_rates:
    weights = np.load(f'vdsp_weights_{LR}_{epoch}.npy')
    assigned_labels = np.load(f'assigned_labels_{LR}_{epoch}.npy')
    
    correct_predictions = 0
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)

    for img_idx, img in tqdm(enumerate(images), total=len(images), desc="Testing SNN"):
        v_pre.fill(0.0)
        v_post.fill(0.0)
        
        neuron_spikes = np.zeros(NUM_OUTPUT)
        
        for t in np.arange(0, T_MAX, DT):
            v_pre += (DT / 0.03) * (-v_pre + img + 0.5)
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            
            v_post += (DT / 0.03) * (-v_post + (spikes_pre.astype(float) @ weights))
            spikes_post = v_post >= 1.0
            
            if spikes_post.any():
                winner = np.argmax(v_post)
                neuron_spikes[winner] += 1
                v_post[:] = 0.0
                
        # Predict class based on the neuron with max spikes
        if neuron_spikes.sum() > 0:
            predicted_neuron = np.argmax(neuron_spikes)
            predicted_label = assigned_labels[predicted_neuron]
            
            if predicted_label == true_labels[img_idx]:
                correct_predictions += 1
            
    final_acc = (correct_predictions / len(images)) * 100
    print(f"Accuracy for LR={LR}: {final_acc:.2f}%")

Testing SNN: 100%|██████████████████| 10000/10000 [00:40<00:00, 248.03it/s]


Accuracy for LR=0.0001: 14.32%


Testing SNN: 100%|██████████████████| 10000/10000 [00:40<00:00, 246.60it/s]


Accuracy for LR=0.0005: 15.26%


Testing SNN: 100%|██████████████████| 10000/10000 [00:40<00:00, 247.75it/s]


...

Accuracy for LR=0.01: 54.10%


Testing SNN: 100%|██████████████████| 10000/10000 [00:38<00:00, 262.37it/s]


Accuracy for LR=0.05: 43.26%


Testing SNN: 100%|██████████████████| 10000/10000 [00:38<00:00, 260.22it/s]

Accuracy for LR=0.1: 36.70%
